**Control Bonds**

In [ ]:
import sys
import os
import numpy as np
import pandas as pd

_HERE = os.getcwd()
_ROOT = os.path.dirname(_HERE)
sys.path.insert(0, _ROOT)
sys.path.insert(0, _HERE)

DATABASE_DIR = os.path.join(_ROOT, 'Database')

bonds = pd.read_parquet(os.path.join(DATABASE_DIR, 'bonds.parquet'))

dates = bonds.index.get_level_values('Date')
print(f'Date range: {dates.min().date()} to {dates.max().date()}')

nan_mask = bonds.isna().any(axis=1)
print(f'Rows with at least one NaN: {nan_mask.sum()} / {len(bonds)}')

nan_by_ticker = (
    bonds[nan_mask]
    .index.to_frame(index=False)
    .groupby('Ticker')['Date']
    .agg(count='count', first='min', last='max')
)
print('\nNaN occurrences by ticker:')
print(nan_by_ticker)


def nan_run_stats(group):
    # position-based (not calendar-day) so weekends/holidays in the
    # ticker's own date index don't look like gaps
    is_nan = group.isna().any(axis=1).to_numpy()
    if not is_nan.any():
        return pd.Series({'nan_count': 0, 'runs': 0, 'longest_run': 0})
    positions = np.flatnonzero(is_nan)
    breaks = np.diff(positions) != 1
    run_ids = np.concatenate(([0], np.cumsum(breaks)))
    run_lengths = np.bincount(run_ids)
    return pd.Series({
        'nan_count': int(is_nan.sum()),
        'runs': int(run_ids.max() + 1),
        'longest_run': int(run_lengths.max()),
    })


nan_pattern = (
    bonds.sort_index()
    .groupby(level='Ticker')
    .apply(nan_run_stats)
)
nan_pattern = nan_pattern[nan_pattern['nan_count'] > 0]
print('\nNaN pattern by ticker (runs=1 -> one consecutive block, runs==nan_count -> scattered/random):')
print(nan_pattern)

bonds
